In [2]:
import os
import cv2  # or use PIL as an alternative
import numpy as np
import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import os
import json
from functools import partial
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, BatchNormalization, Activation, GlobalMaxPooling2D, Dense, SeparableConv2D, Dropout
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras import layers, models
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from sklearn.metrics import accuracy_score
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, MaxPooling2D, Input, concatenate, GlobalAveragePooling2D, AveragePooling2D, Reshape, Multiply
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization
# from keras.models import Sequential
from tensorflow.keras.optimizers import Adam
import tensorflow_model_optimization.sparsity.keras as tfmot_keras
import tempfile
import datetime
import pandas as pd

# Define a log directory
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)


In [3]:
import os
import numpy as np
from PIL import Image, ImageFilter
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical  # You can use np.eye as well for one-hot encoding
from tensorflow.keras.preprocessing.image import array_to_img
import random
import math
from skimage.restoration import inpaint, denoise_wavelet

# notes
# data uses datagen to randomly augment half of dataset if a class has over 1500, otherwise fill
# to 1500 using random augmented images.
# also uses median and denoise wavelet filters (does not harm finer details)

def augment_image(image):
    # Flips the image randomly
    image = tf.image.random_flip_left_right(image)

    # Increase the image size, then randomly crop it down to
    # the original dimensions
    resize_factor = random.uniform(1, 1.2)
    new_height = math.floor(resize_factor * 256)
    new_width = math.floor(resize_factor * 256)
    image = tf.image.resize_with_crop_or_pad(image, new_height, new_width)
    image = tf.image.random_crop(image, size=(256, 256, 3))

    # Vary the brightness of the image
    image = tf.image.random_brightness(image, max_delta=0.2)

    return image

def dull_razor(image):
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Use a blackhat morphological operation to detect dark hair
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)

    # Threshold the blackhat to create a binary hair mask
    _, mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)

    # Inpaint using the mask
    mask = mask.astype(bool)
    image_float = image / 255.0
    result = inpaint.inpaint_biharmonic(image_float, mask, channel_axis=-1)

    return result

def resize_and_convert(image_path, shape=(256, 256), img_type='RGB', denoise=True):
    # Load the image
    img = Image.open(image_path)

    # Resize the image 
    img = img.resize(shape)
    img = img.convert(img_type)

    if denoise:
        img = img.filter(ImageFilter.MedianFilter(size=3))

        # normalize
        img_array = dull_razor(np.array(img)) # normalizes already
        denoised = denoise_wavelet(img_array, channel_axis=-1, 
                                convert2ycbcr=True, 
                                method='BayesShrink', mode='soft')
        
        return np.clip(denoised, 0, 1).astype(np.float32)

    return (np.array(img) / 255.0).astype(np.float32)

def load_images_from_folder(metadata, *folders):
    # Load images into a NumPy array
    images = []
    labels = []

    for folder in folders: # folder is folder path]
        data = pd.read_csv(metadata)
        for i, row in data.iterrows():
            img_path = os.path.join(folder, row["image_id"] + ".jpg")
            try:
                # Read and resize image
                img_array = resize_and_convert(img_path)
                images.append(img_array)
                labels.append(row["dx"])  # dx is the column for diagnosis
            except:
                continue # images might not be in the part that is being searched
    
    return balance(images, labels)

def balance(images, labels):
    images_by_class = {}

    # fill dictionary with the 7 classes
    for img, label in zip(images, labels):
        if label not in images_by_class:
            images_by_class[label] = []
        images_by_class[label].append(img)

    images_balanced = []
    labels_balanced = []

    for label, images in images_by_class.items():
        if len(images) > 1000:
            selected = random.sample(images, 1000)
        else:
            selected = images[:]
            while len(selected) < 1000:
                img_array = random.choice(images)
              
                img = tf.convert_to_tensor(img_array, dtype=tf.float32)
                img_array = img.numpy()
                img_array = (img_array + 1.0) / 2.0 # remove negative numbers

                selected.append(img_array)
        images_balanced.extend(selected)
        labels_balanced.extend([label] * 1000)
    
    return np.array(images_balanced), np.array(labels_balanced)

# paths to directory
path1 = "HAM10000\HAM10000_images_part_1"
path2 = "HAM10000\HAM10000_images_part_2"

metadata_path = "HAM10000\HAM10000_metadata.csv"

images_np, labels_np = load_images_from_folder(metadata_path, path1, path2)

In [10]:
images_np = np.load('images.npy')
labels_np = np.load('labels.npy')

# split into training and validation data
images_train, images_test, labels_train, labels_test = train_test_split(
    images_np, labels_np, test_size=0.15)

# Encode the labels into integers
label_encoder = LabelEncoder()

labels_train_encoded = label_encoder.fit_transform(labels_train)
labels_test_encoded = label_encoder.fit_transform(labels_test)

# One-hot encode the integer labels
labels_train_one_hot = to_categorical(labels_train_encoded)
labels_test_one_hot = to_categorical(labels_test_encoded)

print(f"Images shape: {images_np.shape}")
print(f"One-hot encoded labels shape (training): {labels_train_one_hot.shape}")
print(f"One-hot encoded labels shape (testing): {labels_test_one_hot.shape}")

#tensor_reshaped = tf.reshape(labels_train_one_hot, (-1, 1, 1, 4))

encoded_justValue = tf.reshape(labels_train_encoded, (-1, 1, 1, 1))

Images shape: (7000, 256, 256, 3)
One-hot encoded labels shape (training): (5950, 7)
One-hot encoded labels shape (testing): (1050, 7)


In [6]:
np.save('images.npy', images_np)
np.save('labels.npy', labels_np)